In [1]:
import requests
import csv
import time

In [2]:
base_url = "https://api.openalex.org/works"
params = {
    'filter': 'institutions.country_code:VN,default.search:artificial intelligence',
    'per-page': 200,
    'cursor': '*',
    'mailto': 'student_project@example.com' 
}
csv_file = 'openalex_ai_vietnam.csv'
fieldnames = ['Paper_ID', 'Title', 'Publication_Year', 'Citations', 'University_Name', 'Authors']
max_pages = 50
page = 1

In [3]:
all_papers = []
while params['cursor'] and page <= max_pages:
    response = requests.get(base_url, params=params, timeout=15)
    data = response.json()
    results = data.get('results', [])
    
    if not results:
        break
    
    for work in results:
        paper_id = work.get('id', '').replace('https://openalex.org/', '')
        title = work.get('title', '')
        if not title:
            continue
        
        year = work.get('publication_year', '')
        citations = work.get('cited_by_count', 0)
        
        uni_names = []
        authors = []
        
        for authorship in work.get('authorships', []):
            author_name = authorship.get('author', {}).get('display_name', '')
            if author_name:
                authors.append(author_name)
                
            for inst in authorship.get('institutions', []):
                if inst.get('country_code') == 'VN':
                    uni_name = inst.get('display_name', '')
                    if uni_name and uni_name not in uni_names:
                        uni_names.append(uni_name)
        
        uni_str = " | ".join(uni_names)
        authors_str = ", ".join(authors[:5])
        
        all_papers.append({
            'Paper_ID': paper_id,
            'Title': title,
            'Publication_Year': year,
            'Citations': citations,
            'University_Name': uni_str,
            'Authors': authors_str
        })
    
    params['cursor'] = data.get('meta', {}).get('next_cursor')
    page += 1
    time.sleep(0.1)

In [4]:
# Fetch PTIT explicitly
ptit_params = {
    'filter': 'institutions.id:I4210096823,default.search:artificial intelligence',
    'per-page': 200,
    'mailto': 'student_project@example.com'
}
resp = requests.get(base_url, params=ptit_params)
ptit_data = resp.json()
for work in ptit_data.get('results', []):
    paper_id = work.get('id', '').replace('https://openalex.org/', '')
    title = work.get('title', '')
    if not title: continue
    year = work.get('publication_year', '')
    citations = work.get('cited_by_count', 0)
    uni_names = []
    authors = []
    for authorship in work.get('authorships', []):
        author_name = authorship.get('author', {}).get('display_name', '')
        if author_name: authors.append(author_name)
        for inst in authorship.get('institutions', []):
            if inst.get('country_code') == 'VN':
                uni_name = inst.get('display_name', '')
                if uni_name and uni_name not in uni_names:
                    uni_names.append(uni_name)
    if 'Posts and Telecommunications Institute of Technology' not in uni_names:
        uni_names.append('Posts and Telecommunications Institute of Technology')
    all_papers.append({
        'Paper_ID': paper_id,
        'Title': title,
        'Publication_Year': year,
        'Citations': citations,
        'University_Name': ' | '.join(uni_names),
        'Authors': ', '.join(authors[:5])
    })

In [5]:
with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_papers)